Una solución empresarial completa
Ahora llevaremos nuestro proyecto del día 1 al siguiente nivel
DESAFÍO EMPRESARIAL:
Crear un producto que genere un folleto para una empresa que se utilizará para posibles clientes, inversores y posibles reclutas.

Se nos proporcionará un nombre de empresa y su sitio web principal.

Consulte el final de este cuaderno para ver ejemplos de aplicaciones empresariales del mundo real.

Y recuerde: ¡siempre estoy disponible si tiene problemas o ideas! No dude en comunicarse conmigo.

In [4]:
# imports
# Si esto falla, verifica que esté ejecutándose desde un entorno "activado" con (llms) en el símbolo del sistema

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from pathlib import Path

In [17]:
# Inicialización y constantes and constants

env_path = Path.home() / "work" / ".config" / "llm" / "apis.env"
load_dotenv(env_path, override=False)

api_key = os.getenv('OPENAI_API_KEY')

## Config Ollama
ollama_base_url = os.environ["OLLAMA_BASE_URL"].rstrip("/")
ollama_base_url = os.getenv('OLLAMA_BASE_URL')
#ollama_api_key = os.getenv('OLLAMA_API_KEY')
ollama_model = os.getenv('OLLAMA_MODEL_GEMMA')
ollama_username = os.getenv('OLLAMA_USERNAME')
ollama_password = os.getenv('OLLAMA_PASSWORD')

## Config NVIDIA
nvidia_base_url = os.environ["NVIDIA_BASE_URL"].rstrip("/")
nvidia_api_key = os.environ["NVIDIA_API_KEY"].rstrip("/")
model_nvidia="z-ai/glm-5.2"

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

La clave de API parece buena


In [6]:
# La clase para representar una Página Web

class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado, ahora con enlaces
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "Sin título"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Título de la Web:\n{self.title}\nContenido de la Web:\n{self.text}\n\n"

In [18]:
frog = Website("https://cursos.frogamesformacion.com")
print(frog.get_contents())
frog.links

Título de la Web:
Just a moment...
Contenido de la Web:
Enable JavaScript and cookies to continue




[]

Primer paso: hacer que GPT-4o-mini determine qué enlaces son relevantes
Usar una llamada a gpt-4o-mini para leer los enlaces en una página web y responder en JSON estructurado.
Debería decidir qué enlaces son relevantes y reemplazar los enlaces relativos como "/about" con "https://company.com/about". Usaremos "one shot prompting" en las que proporcionamos un ejemplo de cómo debería responder en la solicitud.

Este es un excelente caso de uso para un LLM, porque requiere una comprensión matizada. Imagínate intentar programar esto sin LLMs analizando la página web: ¡sería muy difícil!

Nota al margen: existe una técnica más avanzada llamada "Salidas estructuradas" en la que requerimos que el modelo responda de acuerdo con una especificación. Cubrimos esta técnica en la Semana 8 durante nuestro proyecto autónomo de inteligencia artificial Agentic.

In [10]:
link_system_prompt = "Se te proporciona una lista de enlaces que se encuentran en una página web. \
Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, \
como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.\n"
link_system_prompt += "Debes responder en JSON como en este ejemplo:"
link_system_prompt += """
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}
"""

In [11]:
print(link_system_prompt)

Se te proporciona una lista de enlaces que se encuentran en una página web. Puedes decidir cuáles de los enlaces serían los más relevantes para incluir en un folleto sobre la empresa, como enlaces a una página Acerca de, una página de la empresa, las carreras/empleos disponibles o páginas de Cursos/Packs.
Debes responder en JSON como en este ejemplo:
{
    "links": [
        {"type": "Pagina Sobre nosotros", "url": "https://url.completa/aqui/va/sobre/nosotros"},
        {"type": "Pagina de Cursos": "url": "https://otra.url.completa/courses"}
    ]
}



In [12]:
def get_links_user_prompt(website):
    user_prompt = f"Aquí hay una lista de enlaces de la página web {website.url} - "
    user_prompt += "Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. \
No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.\n"
    user_prompt += "Links (puede que algunos sean links relativos):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [26]:
print(get_links_user_prompt(frog))

Aquí hay una lista de enlaces de la página web https://cursos.frogamesformacion.com - Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa. Responde con la URL https completa en formato JSON. No incluyas Términos y Condiciones, Privacidad ni enlaces de correo electrónico.
Links (puede que algunos sean links relativos):



In [14]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [15]:
anthropic = Website("https://anthropic.com")
anthropic.links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/policy',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/claude-corps',
 'https://www.anthropic.com/policy-on-the-ai-exponential',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'http://trust.anthropic.com/',
 'https://www.anthropic.com/learn',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://www.anthropic.com/engineering',
 'https://platform.claude.com/docs',
 'https://www.anthropic.com/company',
 'https://www.anthropic.com/careers',
 'https://www.anthropic.com/events',
 'https://www.anthropic.com/news',
 'https://claude.ai',
 'https://claude.com/product/overview',
 'https://claude.com/pricing',
 'https://claude.com/contact-sales',
 'https://www.anthropic.com/claude/mythos',
 'https://www.anthropic.com/claude/fable',
 'https://www.anthropic.com/

In [16]:
#Response with OpenAI
get_links("https://anthropic.com")

{'links': [{'type': 'Página Acerca de',
   'url': 'https://www.anthropic.com/company'},
  {'type': 'Carreras/Empleos', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'Página de Investigación',
   'url': 'https://www.anthropic.com/research'},
  {'type': 'Página de Cursos', 'url': 'https://www.anthropic.com/learn'},
  {'type': 'Página de Noticias', 'url': 'https://www.anthropic.com/news'},
  {'type': 'Página de Eventos', 'url': 'https://www.anthropic.com/events'}]}

In [19]:
# Configure with Nvidia.
client = OpenAI(
    base_url=nvidia_base_url,
    api_key=nvidia_api_key,
)

In [23]:
def get_links_nvidia(url):
    website = Website(url)
    response = client.chat.completions.create(
        model=model_nvidia,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        temperature=1,
        top_p=1,
        max_tokens=16384,
        seed=42,
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [24]:
#Response with OpenAI
get_links_nvidia("https://anthropic.com")

{'links': [{'type': 'Página de Inicio', 'url': 'https://www.anthropic.com/'},
  {'type': 'Página Empresa', 'url': 'https://www.anthropic.com/company'},
  {'type': 'Página Empleos', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'Página Investigación',
   'url': 'https://www.anthropic.com/research'},
  {'type': 'Página Ingeniería',
   'url': 'https://www.anthropic.com/engineering'},
  {'type': 'Página Noticias', 'url': 'https://www.anthropic.com/news'},
  {'type': 'Página Eventos', 'url': 'https://www.anthropic.com/events'},
  {'type': 'Página de Profesiones/Educación',
   'url': 'https://claude.com/solutions/education'},
  {'type': 'Página de Profesiones/Educación',
   'url': 'https://claude.com/solutions/teachers'},
  {'type': 'Página Aprendizaje', 'url': 'https://www.anthropic.com/learn'},
  {'type': 'Página Recursos/Tutoriales',
   'url': 'https://claude.com/resources/tutorials'},
  {'type': 'Página Recursos/Casos de Uso',
   'url': 'https://claude.com/resources/use-cases'}

In [25]:
#Response with OpenAI
get_links_nvidia("https://cursos.frogamesformacion.com")

{'links': [{'type': 'Pagina Sobre nosotros',
   'url': 'https://cursos.frogamesformacion.com/pages/about-us'},
  {'type': 'Pagina de Cursos',
   'url': 'https://cursos.frogamesformacion.com/collections/all-courses'},
  {'type': 'Pagina de Empresa',
   'url': 'https://cursos.frogamesformacion.com'}]}

## Segundo paso: ¡crea el folleto!
Reúne todos los detalles en otro mensaje para GPT4-o y Nvidia model z-ai/glm-5.2

In [27]:
## Function with OpenAI
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Links encontrados:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [28]:
print(get_all_details("https://anthropic.com"))

Links encontrados: {'links': [{'type': 'Página Acerca de', 'url': 'https://www.anthropic.com/company'}, {'type': 'Carreras/Empleos', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Página de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Página de Aprendizaje', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Página de Noticias', 'url': 'https://www.anthropic.com/news'}]}
Landing page:
Título de la Web:
Home \ Anthropic
Contenido de la Web:
Skip to main content
Skip to footer
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Mod

In [29]:
## Function with Nvidia
def get_all_details_nvidia(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links_nvidia(url)
    print("Links encontrados:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [30]:
print(get_all_details_nvidia("https://anthropic.com"))

Links encontrados: {'links': [{'type': 'Pagina de la empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Pagina de carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Investigacion', 'url': 'https://www.anthropic.com/research'}, {'type': 'Noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Producto Claude', 'url': 'https://claude.com/product/overview'}, {'type': 'Documentacion de la plataforma', 'url': 'https://platform.claude.com/docs'}, {'type': 'Precios', 'url': 'https://claude.com/pricing'}, {'type': 'Contacto de ventas', 'url': 'https://claude.com/contact-sales'}, {'type': 'Casos de uso', 'url': 'https://claude.com/resources/use-cases'}, {'type': 'Tutoriales', 'url': 'https://claude.com/resources/tutorials'}, {'type': 'Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Transparencia', 'url': 'https://www.anthropic.com/transparency'}]}
Landing page:
Título de la Web:
Home \ Anthropic
Contenido de la Web:
Skip to main content
Skip t

In [31]:
system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa\
y crea un folleto breve sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
Incluye detalles sobre la cultura de la empresa, los clientes, las carreras/empleos y los cursos/packs para futuros empleos si tienes la información."

# O descomenta las líneas a continuación para obtener un folleto más humorístico: esto demuestra lo fácil que es incorporar el "tono":

# system_prompt = "Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa \
# y crea un folleto breve, divertido y gracioso sobre la empresa para posibles clientes, inversores y nuevos empleados. Responde en formato Markdown.\
#Incluye detalles sobre la cultura de la empresa, los clientes y los cursos/packs para futuros empleos si tienes la información."

In [33]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"Estás mirando una empresa llamada: {company_name}\n"
    user_prompt += f"Aquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncar si tiene más de 20.000 caracteres
    return user_prompt

In [34]:
def get_brochure_user_prompt_nvidia(company_name, url):
    user_prompt = f"Estás mirando una empresa llamada: {company_name}\n"
    user_prompt += f"Aquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\n"
    user_prompt += get_all_details_nvidia(url)
    user_prompt = user_prompt[:20_000] # Truncar si tiene más de 20.000 caracteres
    return user_prompt

In [35]:
get_brochure_user_prompt("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Pagina de la empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina de investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Pagina de eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Pagina de aprendizaje', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Pagina de noticias', 'url': 'https://www.anthropic.com/news'}]}


"Estás mirando una empresa llamada: Anthropic\nAquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\nLanding page:\nTítulo de la Web:\nHome \\ Anthropic\nContenido de la Web:\nSkip to main content\nSkip to footer\nResearch\nPolicy\nCommitments\nInitiatives\nClaude's Constitution\nClaude Corps\nPolicy on the AI Exponential\nTransparency\nResponsible Scaling Policy\nTrust center\nSecurity and compliance\nLearn\nLearn\nAnthropic Academy\nTutorials\nUse cases\nEngineering at Anthropic\nDeveloper docs\nCompany\nAbout\nCareers\nEvents\nNews\nTry Claude\nTry Claude\nTry Claude\nLearn more about Claude\nAbout Claude\nOverview\nPricing\nContact sales\nModels\nMythos\nFable\nOpus\nSonnet\nHaiku\nLog in\nClaude.ai\nClaude Console\nEN\nThis is some text inside of a div block.\nLog in to Claude\nLog in to Claude\nLog in to Claude\nDownload app\nDownload app\nDownload app\nResearch\nPolicy\nCommi

In [36]:
get_brochure_user_prompt_nvidia("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Pagina Sobre nosotros', 'url': 'https://www.anthropic.com/company'}, {'type': 'Pagina de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina de Investigacion', 'url': 'https://www.anthropic.com/research'}, {'type': 'Pagina de Producto', 'url': 'https://claude.com/product/overview'}, {'type': 'Pagina de Precios', 'url': 'https://claude.com/pricing'}, {'type': 'Pagina de Politicas', 'url': 'https://www.anthropic.com/policy'}, {'type': 'Pagina de Aprendizaje', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Pagina de Noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Pagina de Ingenieria', 'url': 'https://www.anthropic.com/engineering'}, {'type': 'Pagina de Transparencia', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'Pagina de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Pagina de Contacto', 'url': 'https://claude.com/contact-sales'}]}


"Estás mirando una empresa llamada: Anthropic\nAquí se encuentra el contenido de su página de inicio y otras páginas relevantes; usa esta información para crear un breve folleto de la empresa en Markdown.\nLanding page:\nTítulo de la Web:\nHome \\ Anthropic\nContenido de la Web:\nSkip to main content\nSkip to footer\nResearch\nPolicy\nCommitments\nInitiatives\nClaude's Constitution\nClaude Corps\nPolicy on the AI Exponential\nTransparency\nResponsible Scaling Policy\nTrust center\nSecurity and compliance\nLearn\nLearn\nAnthropic Academy\nTutorials\nUse cases\nEngineering at Anthropic\nDeveloper docs\nCompany\nAbout\nCareers\nEvents\nNews\nTry Claude\nTry Claude\nTry Claude\nLearn more about Claude\nAbout Claude\nOverview\nPricing\nContact sales\nModels\nMythos\nFable\nOpus\nSonnet\nHaiku\nLog in\nClaude.ai\nClaude Console\nEN\nThis is some text inside of a div block.\nLog in to Claude\nLog in to Claude\nLog in to Claude\nDownload app\nDownload app\nDownload app\nResearch\nPolicy\nCommi

In [37]:
## Create brochure with OpenAI
def create_brochure_ollama(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [38]:
create_brochure_ollama("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Página Acerca de', 'url': 'https://www.anthropic.com/company'}, {'type': 'Página de Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Página de Investigación', 'url': 'https://www.anthropic.com/research'}, {'type': 'Página de Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Página de Noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Página de Aprendizaje', 'url': 'https://www.anthropic.com/learn'}]}


# Anthropic: Transforming AI with Safety and Responsibility

## Acerca de Anthropic
Anthropic es una corporación de beneficio público dedicada a la investigación y desarrollo de sistemas de inteligencia artificial (IA) que sean fiables, interpretables y manejables. Nuestra misión es construir sistemas de IA que las personas puedan confiar, realizando investigaciones sobre las oportunidades y riesgos asociados.

### Propósito
Creemos que la IA tendrá un impacto profundo en el mundo. Nos dedicamos a desarrollar tecnología que sea una fuerza positiva a largo plazo para la humanidad.

### Cultura empresarial
Nuestra cultura está basada en principios que priorizan el bien global, la amabilidad hacia los usuarios y la transparencia en nuestros procesos. Fomentamos una **"carrera hacia la cima"** en términos de seguridad, donde los desarrolladores de IA compiten en crear sistemas seguros y confiables. Valorizamos la colaboración interdisciplinaria y el aprendizaje continuo.

## Soluciones y Productos
Anthropic desarrolla una gama de productos bajo la plataforma **Claude**, diseñado para ser útil, honesto y inofensivo. Algunos de estos incluyen:
- **Claude**: Un asistente de IA general.
- **Claude Code**: Herramientas específicas para la programación.
- **Claude Science**: Aplicaciones para investigadores.
- **Claude Security**: Enfoque en la ciberseguridad.

## Clientes
Trabajamos con una variedad de sectores, incluyendo:
- Gobierno
- Servicios de salud
- Educación (K-12 y superior)
- Empresas y pequeñas empresas
- Organizaciones sin fines de lucro

De esta manera, nuestros productos están diseñados para beneficiar a una amplia gama de usuarios, desde empresas hasta grupos de la sociedad civil.

## Oportunidades de Carrera 
En Anthropic, buscamos personas que tengan un enfoque integral hacia los problemas difíciles de la IA. Nuestros roles abarcan un amplio espectro de habilidades, desde investigación y desarrollo hasta políticas y operaciones.

### Beneficios
Ofrecemos un paquete de beneficios completo que incluye:
- Seguros de salud, dental y de visión.
- Licencia parental remunerada de hasta 22 semanas.
- Estipendios de bienestar.
- Programas de educación continua.

### Proceso de Contratación
Valoramos las habilidades prácticas sobre las credenciales formales. Nuestro proceso de entrevistas es accesible y busca comprender la experiencia y la motivación del candidato.

## Compromiso con la Educación
### Anthropic Academy
Nos comprometemos a educar sobre el uso seguro de la IA. Ofrecemos cursos y tutoriales a través de **Anthropic Academy** para ayudar a los usuarios a navegar por el panorama de IA, preparándolos para futuros empleos y desafíos en el campo.

### Oportunidades de Aprendizaje
- Tutoriales y recursos sobre el desarrollo y uso de nuestros modelos de IA.
- Estudio de casos y programas enfocados en aplicaciones prácticas de la IA.

## Únete a Nosotros
Si te apasiona la tecnología y la seguridad de la IA, ¡te invitamos a que te unas a nuestro equipo! En Anthropic, estamos en la vanguardia del futuro de la inteligencia artificial. 

Visita [Anthropic Careers](https://www.anthropic.com/careers) para explorar nuestras oportunidades disponibles y formar parte de una misión que busca cambiar el mundo para mejor.

¡Estamos emocionados de construir el futuro contigo!

In [39]:
## Create brochure with Nvidia
def create_brochure_nvidia(company_name, url):
    response = client.chat.completions.create(
    model=model_nvidia,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt_nvidia(company_name, url)}
          ],
        temperature=1,
        top_p=1,
        max_tokens=16384,
        seed=42,
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [40]:
create_brochure_nvidia("Anthropic", "https://anthropic.com")

Links encontrados: {'links': [{'type': 'Pagina Empresa', 'url': 'https://www.anthropic.com/company'}, {'type': 'Pagina Carreras', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Pagina Producto', 'url': 'https://claude.com/product/overview'}, {'type': 'Pagina Precios', 'url': 'https://claude.com/pricing'}, {'type': 'Pagina Investigacion', 'url': 'https://www.anthropic.com/research'}, {'type': 'Pagina Ingenieria', 'url': 'https://www.anthropic.com/engineering'}, {'type': 'Pagina Aprendizaje', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Pagina Noticias', 'url': 'https://www.anthropic.com/news'}, {'type': 'Pagina Eventos', 'url': 'https://www.anthropic.com/events'}, {'type': 'Pagina Clientes', 'url': 'https://www.claude.com/customers'}, {'type': 'Pagina Contacto Ventas', 'url': 'https://claude.com/contact-sales'}]}


# Folleto Informativo: Anthropic
*Investigación y productos de IA que priorizan la seguridad en la frontera tecnológica.*

---

## 🏢 Sobre la Empresa
**Anthropic** es una corporación de beneficio público (Public Benefit Corporation) dedicada a la investigación y seguridad en Inteligencia Artificial. Nuestra misión es crear sistemas de IA confiables, interpretables y direccionables que sirvan al bienestar a largo plazo de la humanidad. Creemos que la IA tendrá un impacto inmenso en el mundo y trabajamos activamente para asegurar sus beneficios y mitigar sus riesgos.

## 🤝 Nuestra Cultura y Valores
En Anthropic, damos forma al futuro de la IA asumiendo la responsabilidad de guiar al mundo a través de una revolución tecnológica. Nuestro equipo interdisciplinario —compuesto por investigadores, ingenieros, expertos en políticas y líderes de operaciones— se guía por 7 principios fundamentales:

1. **Actuar por el bien global:** Tomamos decisiones audaces para maximizar los resultados positivos para la humanidad a largo plazo.
2. **Mantener el equilibrio (Luz y Sombra):** Entendemos los riesgos potenciales de la IA ("sombra") mientras trabajamos para lograr beneficios sin precedentes ("luz").
3. **Ser buenos con nuestros usuarios:** Fomentamos la generosidad y la amplitud de miras con clientes, colegas y cualquier persona afectada por nuestra tecnología.
4. **Encender una "carrera hacia la cima" en seguridad:**Competimos para establecer el estándar más alto en sistemas de IA seguros, empujando a la industria a hacer lo mismo.
5. **Hacer lo simple que funcione:** Nos enfocamos en el impacto real y empírico, iterando desde las soluciones más sencillas.
6. **Ser útiles, honestos e inofensivos:** Somos una organización de alta confianza y bajo ego; la colaboración y la comunicación directa son clave.
7. **La misión primero:** Nuestra misión compartida nos permite actuar con rapidez y propósito, asumiendo todos la responsabilidad de nuestro éxito.

## 🧑‍💼 Clientes y Soluciones
Ofrecemos **Claude**, nuestra familia de modelos de IA (Mythos, Fable, Opus, Sonnet, Haiku) diseñada para ser útil, honesta e inofensiva. Colaboramos con la sociedad civil, gobiernos, academia y empresas para proporcionar herramientas prácticas.

### Productos Destacados:
*   **Claude.ai, Claude Console y Claude Code:** Herramientas versátiles para el día a día y desarrollo.
*   **Suites Especializadas:** Claude Science, Claude Security, Claude Design y Claude Cowork.
*   **Integraciones:** Extensiones para Chrome, Slack y Microsoft 365.

### Sectores Atendidos:
Damos soporte a una amplia variedad de industrias con soluciones personalizadas: Agentes de IA, Modernización de código, Ciberseguridad, Servicios Financieros, Gobierno, Salud, Educación Superior, Ciencias de la Vida, Sector Legal, Empresas Sin Fines de Lucro y Pequeñas Empresas.

## 💼 Carreras y Empleo
¿Te atraen los problemas difíciles con consecuencias reales? En Anthropic buscamos personas excepcionales que compartan nuestra visión de seguridad.

### Cómo contratamos:
*   **Enfoque en habilidades:** Nos importa lo que puedes hacer, no dónde aprendiste a hacerlo. La mitad de nuestro equipo técnico no tenía experiencia previa en ML y muchos no tienen título universitario. Valoramos la investigación independiente, los blogs técnicos y el código open source.
*   **Roles Técnicos:** Los ingenieros investigan y los investigadores programan. Fomentamos un ambiente donde todos contribuyen al rumbo de la empresa.
*   **Roles No Técnicos:** Buscamos perfiles que aporten claridad, criterio y un interés genuino en la misión para nuestro equipo de políticas, operaciones y negocios.

### Beneficios de ser parte de Anthropic ("Ants"):
*   **Salud y Bienestar:** Seguro médico, dental y de visión integral. 22 semanas de licencia parental pagada y apoyo en salud mental.
*   **Compensación y Apoyo:** Salarios competitivos, planes de retiro y opción de donación de capital (equity) con matched 1:1 hasta el 25%.
*   **Vida en la Oficina:** Subsidio mensual de bienestar de $500, comidas diarias en la oficina, estipendio para equipo de home office y apoyo para reubicación.

## 📚 Aprendizaje y Desarrollo (Anthropic Academy)
Ofrecemos recursos para que los clientes y futuros profesionales aprendan a sacar el máximo provecho de nuestros modelos:

*   **Anthropic Academy:** Plataforma de aprendizaje y nuestra **Constitución de Claude**, que detalla cómo nuestro modelo toma decisiones éticas.
*   **Recursos Educativos:** Documentación para desarrolladores, tutoriales, casos de uso y artículos sobre ingeniería en Anthropic.
*   **Cursos y Packs:** Contenidos formativos y guías de uso orientadas a la integración de IA segura en flujos de trabajo profesionales y empresariales.

---
*Para explorar oportunidades, soluciones o aprender más sobre nuestra investigación, visita nuestro sitio oficial.*